# Time Series Analysis of Store Data

- [Forecasting with ML and Stuff](https://www.kaggle.com/code/ryanholbrook/forecasting-with-machine-learning/tutorial)
- [Kaggle Page](https://www.kaggle.com/competitions/store-sales-time-series-forecasting/data)


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import torch

from time_series.store_sales import StoreData

In [ ]:
device = (
    torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
)
device

# Notes

From the information
- Wages in the public sector are paid every two weeks on the 15 th and on the last day of the month. Supermarket sales could be affected by this.
- A magnitude 7.8 earthquake struck Ecuador on April 16, 2016. People rallied in relief efforts donating water and other first need products which greatly affected supermarket sales for several weeks after the earthquake.

# Load and Explore Data

In [ ]:
store_data = StoreData()
store_data.train

In [ ]:
input_pivot = store_data.train.pivot(
    columns=("store_nbr", "family"), values="sales"
).sort_index(axis="columns")
input_pivot

# Model Design

- Input data is essentially `date x store_nbr x family`, which is easily tensor-able. The overall data will be `batch, date_len, store_nbr, family`. 
  - Remember to use `batch_first=True` for any LSTM or whatever models. Apparently having the batch dimension be second is an old convention.
- We need to predict 2017-08-16 to 2017-08-31, so 16 day prediction window
- Input window length will be a parameter, sent to the dataset and model
- loss is Root-Mean-Squared-Logarithmic-Error (given by problem)
  - $\text{RMSE}(\log(1+\hat{y}), \log(1+y))$

# Set Up dataset

## Pandas -> Tensor

In [ ]:
store_data.sales_tensor.shape, store_data.families

In [ ]:
# Verify data transform is correct
fig, ax_list = plt.subplots(nrows=2, sharex=True)

store_id = 5
family = "PREPARED FOODS"

pd_data = input_pivot[(store_id, family)]
pt_data = pd.Series(
    store_data.sales_tensor[
        :, store_id - 1, store_data.families.get_loc(family)
    ].numpy(),
    index=input_pivot.index,
)

fig.suptitle(f"{store_id=}, '{family}'")
pd_data.plot(ax=ax_list[0], grid=True, label="pandas")
pt_data.plot(ax=ax_list[0], grid=True, label="tensor")
ax_list[0].legend()
(pd_data - pt_data).plot(ax=ax_list[1], grid=True, label="diff", legend=True)

## Dataset

In [ ]:
store_data

# Simple Baseline

I dunno, windowed average?